# Contrastive Learning — Framework Examples

Contrastive learning is a self-supervised technique where a model learns by comparing sample pairs — pulling **positives** (similar items) closer and pushing **negatives** (dissimilar items) apart in embedding space.

| Framework | Domain | Key Idea |
|---|---|---|
| **SimCLR** | Vision | NT-Xent loss on augmented image pairs |
| **MoCo** | Vision | Momentum encoder + queue of negatives |
| **CLIP** | Vision-Language | Align image & text embeddings jointly |
| **DPR / Bi-Encoder** | NLP / Retrieval | Dense query-document retrieval |
| **SupCon** | Any | Supervised labels define positive sets |

Each section below provides a minimal, runnable example demonstrating the core mechanism.

In [0]:
%pip install -q sentence-transformers
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 1. SimCLR

**Key idea:** Given a batch of images, apply two *different* random augmentations to each to produce two views (a positive pair). All other images in the batch act as negatives. The **NT-Xent (Normalized Temperature-scaled Cross Entropy)** loss maximises agreement between positive pairs while repelling negatives.

**Architecture:**
- `Encoder` → backbone (e.g., ResNet) that maps input → representation `h`
- `Projection Head` → MLP that maps `h → z` (embeddings used for loss)
- After training, discard projection head; use `h` for downstream tasks

**NT-Xent Loss formula:** For a positive pair `(i, j)`:
$$\mathcal{L}_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j)/\tau)}{\sum_{k \neq i} \exp(\text{sim}(z_i, z_k)/\tau)}$$

In [0]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── NT-Xent Loss ─────────────────────────────────────────────────────────────
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, z_i, z_j):
        """z_i, z_j: L2-normalised embeddings, shape [N, D]"""
        N = z_i.size(0)
        z = torch.cat([z_i, z_j], dim=0)                                       # [2N, D]
        sim = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)        # [2N, 2N]  [2N, 1, D] [1, 2N, D]
        sim = sim / self.temperature

        # Remove self-similarity (diagonal)
        mask = torch.eye(2 * N, dtype=torch.bool, device=z.device)
        sim.masked_fill_(mask, float('-inf'))

        # Positive pair for sample i is sample (N+i), and vice-versa
        labels = torch.cat([torch.arange(N, 2*N), torch.arange(N)]).to(z.device)
        return F.cross_entropy(sim, labels)


# ── Encoder + Projection Head ─────────────────────────────────────────────────
class SimCLRModel(nn.Module):
    def __init__(self, input_dim=128, hidden_dim=64, proj_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.projector = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, proj_dim)
        )

    def forward(self, x):
        h = self.encoder(x)               # representation (kept after training)
        z = self.projector(h)             # projection (used only during training)
        return F.normalize(z, dim=-1)


# ── Training Loop ─────────────────────────────────────────────────────────────
torch.manual_seed(42)
batch_size, input_dim = 64, 128
model     = SimCLRModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = NTXentLoss(temperature=0.5)

print("Training SimCLR (NT-Xent loss) — 10 steps")
for step in range(10):
    x     = torch.randn(batch_size, input_dim)
    # Two augmented views: slight Gaussian noise simulates augmentation
    x_i   = x + 0.1 * torch.randn_like(x)
    x_j   = x + 0.1 * torch.randn_like(x)

    z_i, z_j = model(x_i), model(x_j)
    loss  = criterion(z_i, z_j)

    optimizer.zero_grad(); loss.backward(); optimizer.step()
    print(f"  Step {step+1:2d} | NT-Xent Loss: {loss.item():.4f}")

print("\n✓ SimCLR done. Encoder learned representations without any labels.")
print(f"  Embedding dim (z): 32 | Representation dim (h): 64")

## 2. MoCo (Momentum Contrast)

**Problem with SimCLR:** Needs very large batches for enough negatives (e.g., 4096+ images).

**MoCo's solution:** Maintain a **queue** of past encoded keys as negatives, decoupled from batch size. Use a **momentum encoder** (slow-moving copy of the query encoder) to produce consistent key representations.

**Key components:**

| Component | Role |
|---|---|
| Query encoder `f_q` | Encodes the current sample; updated by gradient |
| Key encoder `f_k` | Momentum-updated copy of `f_q`; **no gradients** |
| Queue | Stores K recent key embeddings as negatives |

**Momentum update:**  
`f_k ← m · f_k + (1 - m) · f_q`  (m ≈ 0.999, so f_k changes slowly)

**Loss:** InfoNCE — query `q` should match its paired key `k+`, not the K queue entries.

In [0]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy


# ── Small encoder backbone ────────────────────────────────────────────────────
class BaseEncoder(nn.Module):
    def __init__(self, input_dim=128, hidden_dim=64, proj_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, proj_dim)
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


# ── MoCo ──────────────────────────────────────────────────────────────────────
class MoCo(nn.Module):
    def __init__(self, input_dim=128, proj_dim=32, queue_size=256,
                 momentum=0.999, temperature=0.07):
        super().__init__()
        self.m, self.T, self.K = momentum, temperature, queue_size

        self.encoder_q = BaseEncoder(input_dim, 64, proj_dim)      # query encoder
        self.encoder_k = copy.deepcopy(self.encoder_q)             # key encoder (no grad)
        for p in self.encoder_k.parameters():
            p.requires_grad = False

        # Circular queue of negatives
        self.register_buffer("queue", F.normalize(torch.randn(proj_dim, queue_size), dim=0))
        self.register_buffer("queue_ptr", torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def _momentum_update(self):
        """Exponential moving average: f_k <- m*f_k + (1-m)*f_q"""
        for p_q, p_k in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            p_k.data = p_k.data * self.m + p_q.data * (1.0 - self.m)

    @torch.no_grad()
    def _dequeue_enqueue(self, keys):
        ptr = int(self.queue_ptr)
        self.queue[:, ptr:ptr + keys.shape[0]] = keys.T
        self.queue_ptr[0] = (ptr + keys.shape[0]) % self.K

    def forward(self, x_q, x_k):
        q = self.encoder_q(x_q)                        # [N, D]
        with torch.no_grad():
            self._momentum_update()                    # update key encoder
            k = self.encoder_k(x_k)                    # [N, D], no gradient

        # InfoNCE: positive = (q, k), negatives = queue
        l_pos = torch.einsum('nd,nd->n', q, k).unsqueeze(-1)              # [N, 1]
        l_neg = torch.einsum('nd,dk->nk', q, self.queue.clone().detach()) # [N, K]
        logits = torch.cat([l_pos, l_neg], dim=1) / self.T                # [N, K+1]
        labels = torch.zeros(logits.size(0), dtype=torch.long)            # positive is index 0

        loss = F.cross_entropy(logits, labels)
        self._dequeue_enqueue(k)
        return loss


# ── Training Loop ─────────────────────────────────────────────────────────────
torch.manual_seed(42)
moco      = MoCo(input_dim=128, proj_dim=32, queue_size=256, momentum=0.999)
optimizer = torch.optim.SGD(moco.parameters(), lr=0.03, momentum=0.9, weight_decay=1e-4)

print("Training MoCo (InfoNCE loss, queue size=256) — 10 steps")
for step in range(10):
    x   = torch.randn(32, 128)
    x_q = x + 0.1 * torch.randn_like(x)   # query view
    x_k = x + 0.1 * torch.randn_like(x)   # key view

    loss = moco(x_q, x_k)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    print(f"  Step {step+1:2d} | InfoNCE Loss: {loss.item():.4f}")

print("\n✓ MoCo done. Key encoder updated via momentum — gradient flows only through query encoder.")
print(f"  Queue stores 256 negatives, independent of batch size (32 here).")

## 3. CLIP (Contrastive Language-Image Pretraining)

**Key idea:** Train two separate encoders — one for images, one for text — to produce embeddings in a **shared vector space**. Positive pairs are `(image, its caption)`; all other image-text combinations in the batch are negatives.

**Training signal:**
- An image of a cat should be close to the text `"a photo of a cat"`
- It should be far from `"a photo of a dog"`, `"a red car"`, etc.

**Zero-shot classification:** At test time, compute similarity between the image embedding and text prompts like `"a photo of a {class}"` for each class, and pick the closest.

**Architecture:**
```
Image  → [Image Encoder (ViT/CNN)]  → image_emb  [N, D]
Text   → [Text Encoder (Transformer)] → text_emb   [N, D]
                 ↓
    Symmetric cross-entropy over NxN similarity matrix
```

The example below implements a minimal CLIP from scratch using synthetic image (flat vector) and text (token ids) inputs.

In [0]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ── Image Encoder ─────────────────────────────────────────────────────────────
class ImageEncoder(nn.Module):
    def __init__(self, img_dim=64, proj_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(img_dim, 128), nn.GELU(),
            nn.Linear(128, proj_dim)
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)  ## norm 2 normalisation


# ── Text Encoder ──────────────────────────────────────────────────────────────
class TextEncoder(nn.Module):
    def __init__(self, vocab_size=200, embed_dim=64, proj_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.proj = nn.Linear(embed_dim, proj_dim)
    def forward(self, token_ids):
        # Mean-pool over token positions
        h = self.embedding(token_ids).mean(dim=1)  # [B, embed_dim]
        return F.normalize(self.proj(h), dim=-1)


# ── CLIP Model ────────────────────────────────────────────────────────────────
class CLIP(nn.Module):
    def __init__(self, vocab_size=200, embed_dim=64, img_dim=64, proj_dim=64):
        super().__init__()
        self.img_enc  = ImageEncoder(img_dim, proj_dim)
        self.txt_enc  = TextEncoder(vocab_size, embed_dim, proj_dim)
        # Learnable temperature parameter (initialised to 1/0.07 ~ 14.3)
        self.logit_scale = nn.Parameter(torch.ones([]) * torch.log(torch.tensor(1 / 0.07)))

    def forward(self, images, texts):
        img_emb = self.img_enc(images)                   # [N, D]
        txt_emb = self.txt_enc(texts)                    # [N, D]
        scale   = self.logit_scale.exp().clamp(max=100)  # prevent explosion

        # Symmetric NxN similarity matrix
        logits_img2txt = scale * img_emb @ txt_emb.T     # [N, N]
        logits_txt2img = logits_img2txt.T                 # [N, N]

        # Ground truth: diagonal entries are the positives
        N      = images.size(0)
        labels = torch.arange(N, device=images.device)
        loss   = (F.cross_entropy(logits_img2txt, labels) +
                  F.cross_entropy(logits_txt2img, labels)) / 2
        return loss, logits_img2txt


# ── Training Loop ─────────────────────────────────────────────────────────────
torch.manual_seed(42)
batch_size = 16
clip_model = CLIP(vocab_size=200, embed_dim=64, img_dim=64, proj_dim=64)
optimizer  = torch.optim.Adam(clip_model.parameters(), lr=1e-3)

print("Training CLIP (symmetric image-text contrastive loss) — 10 steps")
for step in range(10):
    images = torch.randn(batch_size, 64)                       # synthetic flat image features
    texts  = torch.randint(0, 200, (batch_size, 12))           # synthetic token id sequences

    loss, logits = clip_model(images, texts)

    # Image → Text retrieval accuracy (is the correct text top-1 for each image?)
    preds  = logits.argmax(dim=1)
    labels = torch.arange(batch_size)
    acc    = (preds == labels).float().mean().item()

    optimizer.zero_grad(); loss.backward(); optimizer.step()
    print(f"  Step {step+1:2d} | Loss: {loss.item():.4f} | Image→Text Acc: {acc:.2f}")

print("\n✓ CLIP done. Shared image-text embedding space learned.")
print("  At inference: compute sim(image_emb, text_emb) for zero-shot classification.")

## 4. DPR / Bi-Encoder — Dense Passage Retrieval

**Key idea:** Two separate encoders — a **query encoder** and a **passage encoder** — map queries and documents into the same dense vector space. Retrieval is a **nearest-neighbour search** over document embeddings.

**Training:** Contrastive learning with `(query, positive_passage)` pairs. In-batch negatives are all other passages in the batch.

**Advantage over BM25 (keyword search):** Can match semantically similar terms even without exact lexical overlap.

> Example: query `"wireless headphones"` can retrieve `"Sony Bluetooth Headset"` even though neither `"wireless"` nor `"headphones"` appear in the title.

**Retrieval pipeline:**
```
Offline: encode all corpus docs → build FAISS / vector index
Online:  encode query → ANN search → top-K results
```

The example below uses `sentence-transformers` (`all-MiniLM-L6-v2`) for an e-commerce search demo.

In [0]:
from sentence_transformers import SentenceTransformer, util
import torch

# Load a compact, pre-trained bi-encoder (~22 MB)
print("Loading bi-encoder model (all-MiniLM-L6-v2)...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# ── E-Commerce product corpus (Croma-style) ──────────────────────────────────
corpus = [
    "Sony WH-1000XM5 Bluetooth Wireless Headphones with Active Noise Cancellation",
    "Samsung 65 inch QLED 4K Smart TV with Quantum Dot Display",
    "Apple MacBook Air M2 13-inch Laptop 8GB RAM 256GB SSD",
    "Voltas 1.5 Ton 3 Star Inverter Split Air Conditioner",
    "boAt Rockerz 450 Bluetooth On-Ear Headset with 15H Battery",
    "LG 43 inch Full HD Smart TV with webOS",
    "Lenovo IdeaPad Slim 5 Laptop 16GB RAM 512GB SSD",
    "Daikin 1 Ton 5 Star Fixed Speed Split AC White",
    "OnePlus Bullets Wireless Z2 Bluetooth Neckband Earphones",
    "TCL 55 inch 4K UHD Android Smart LED TV",
    "HP Pavilion 15 Intel Core i5 Laptop 8GB RAM",
    "Hitachi 1.5 Ton 3 Star Inverter Window AC",
]

# ── User queries ─────────────────────────────────────────────────────────────
queries = [
    "wireless headphones",
    "4K television",
    "laptop for students",
    "air conditioner 1.5 ton",
]

# ── Encode corpus (offline indexing) ──────────────────────────────────────────
print("Encoding product corpus...")
corpus_emb = model.encode(corpus, convert_to_tensor=True, show_progress_bar=False)
print(f"Corpus embeddings: {corpus_emb.shape}  ({len(corpus)} products x {corpus_emb.shape[1]} dims)\n")

# ── Online retrieval ─────────────────────────────────────────────────────────────
print("=" * 60)
print("Dense Retrieval Results (Top-3)")
print("=" * 60)
for query in queries:
    q_emb   = model.encode(query, convert_to_tensor=True)
    scores  = util.cos_sim(q_emb, corpus_emb)[0]         # cosine similarity
    top_k   = torch.topk(scores, k=3)

    print(f"\nQuery : '{query}'")
    for rank, (score, idx) in enumerate(zip(top_k.values, top_k.indices), 1):
        print(f"  #{rank} [{score:.3f}] {corpus[idx]}")

print("\n✓ DPR done. Dense retrieval finds semantic matches even without exact keyword overlap.")

## 5. SupCon — Supervised Contrastive Learning

**Key difference from SimCLR/MoCo:** Instead of treating *only augmented pairs* of the same image as positives, SupCon uses **class labels** to define the positive set. All samples from the same class are positives for each other.

**Why it matters:**
- SimCLR has 1 positive per anchor (the other augmented view)
- SupCon has `|class_size - 1|` positives per anchor → denser, richer gradient signal
- Typically outperforms cross-entropy fine-tuning on downstream tasks

**Loss formula:**
$$\mathcal{L}_{sup} = \sum_{i} \frac{-1}{|P(i)|} \sum_{p \in P(i)} \log \frac{\exp(z_i \cdot z_p / \tau)}{\sum_{a \neq i} \exp(z_i \cdot z_a / \tau)}$$

where $P(i)$ is the set of all indices with the same label as $i$.

**Typical 2-stage training:**
1. Train encoder with SupCon loss (no classification head)
2. Freeze encoder, train a linear classifier on top

In [0]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ── Supervised Contrastive Loss ───────────────────────────────────────────────────
class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss (Khosla et al., NeurIPS 2020).
    All same-class samples form the positive set (not just augmented pairs).
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        """
        features : [N, D]  (L2-normalised embeddings)
        labels   : [N]     (class indices)
        """
        N      = features.size(0)
        device = features.device

        # Pairwise cosine similarity (already normalised)
        sim = torch.matmul(features, features.T) / self.temperature  # [N, N]

        # Positive mask: same class, excluding self
        label_col  = labels.unsqueeze(1)                              # [N, 1]
        mask_pos   = (label_col == label_col.T).float().to(device)    # [N, N]
        mask_self  = torch.eye(N, dtype=torch.bool, device=device)
        mask_pos.masked_fill_(mask_self, 0.0)                         # zero out diagonal

        # Denominator: sum over all pairs except self
        sim_exp    = torch.exp(sim)
        sim_exp.masked_fill_(mask_self, 0.0)
        log_denom  = torch.log(sim_exp.sum(dim=1, keepdim=True) + 1e-8)  # [N, 1]

        # Numerator: sum of log-probs over positives
        log_prob   = sim - log_denom                                  # [N, N]
        n_pos      = mask_pos.sum(dim=1).clamp(min=1)                 # avoid /0
        loss       = -(mask_pos * log_prob).sum(dim=1) / n_pos
        return loss.mean()


# ── Encoder ─────────────────────────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=128, proj_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, proj_dim)
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


# ── Training Loop ─────────────────────────────────────────────────────────────
torch.manual_seed(42)
n_classes   = 5
samples_per = 4       # samples per class in each batch → 20 total
encoder     = Encoder(input_dim=64, hidden_dim=128, proj_dim=32)
optimizer   = torch.optim.Adam(encoder.parameters(), lr=1e-3)
criterion   = SupConLoss(temperature=0.07)

# Pre-define class centres to simulate clustered data
class_centers = torch.randn(n_classes, 64)

print("Training with SupCon Loss — 10 steps")
for step in range(10):
    labels = torch.repeat_interleave(torch.arange(n_classes), samples_per)  # [20]
    x      = class_centers[labels] + 0.3 * torch.randn(len(labels), 64)     # noisy clusters

    features = encoder(x)
    loss     = criterion(features, labels)

    optimizer.zero_grad(); loss.backward(); optimizer.step()
    print(f"  Step {step+1:2d} | SupCon Loss: {loss.item():.4f}")

# ── Evaluate: does the encoder cluster same-class samples? ─────────────────────
with torch.no_grad():
    x_eval   = class_centers[labels] + 0.2 * torch.randn(len(labels), 64)
    feats    = encoder(x_eval)
    sim_mat  = torch.matmul(feats, feats.T)
    sim_mat.fill_diagonal_(-1)               # exclude self
    top1_idx = sim_mat.argmax(dim=1)         # nearest neighbour
    nn_acc   = (labels[top1_idx] == labels).float().mean()

print(f"\n✓ SupCon done.")
print(f"  Nearest-neighbour class accuracy: {nn_acc:.2f}")
print("  (1.0 = each sample's closest neighbour always belongs to the same class)")